# **2.0 RAG for Item Generation**
### Grounding generated items in your own curriculum and source material

**What you'll build.** A retrieval-augmented item generator: a small curriculum corpus with
metadata, a local vector index, a retriever you actually *measure*, and a generation chain that
returns a typed item together with the passage that supports its key plus an automatic check.

| | |
|---|---|
| **Time** | ~35 minutes |
| **Keys needed** | `GOOGLE_API_KEY` (Colab Secrets panel, key icon in the left sidebar) |

Nothing here needs Google Drive, an upload, or a file on your machine. The corpus ships inside
the notebook. If you want to use your own PDFs, there is a flag for that in Stage 1.

### **What You'll Learn:**
- The theoretical foundation of RAG and its applications in educational assessment
- How to build a complete RAG pipeline using open-source tools
- Best practices for generating high-quality, curriculum-aligned assessment items
- Quality assurance and evaluation frameworks for AI-generated content
- Ethical considerations and limitations in automated item generation

### **Why RAG for Item Development?**
Traditional AI language models can "hallucinate" or generate content that sounds plausible but isn't grounded in verified educational standards. RAG solves this by:

**Retrieving** relevant content from authoritative sources (curriculum standards, textbooks, learning objectives)  
**Augmenting** the language model with this context  
**Generating** assessment items that are both creative and factually accurate

Think of it as giving the AI a "reference library" before it writes your exam questions thereby ensuring every generated item is anchored to verified content.

## 2.1 Pipeline Overview

A typical RAG pipeline consists of several stages.  Each stage plays a distinct role in making sure that the final generated item reflects accurate curriculum content and is easy for educators to trust:

1. **Data ingestion and document preparation** – gather curricular materials and convert them into a uniform format that the computer can process, e.g, PDFs.

2. **Splitting the documents (“chunking”)** – long texts are divided into smaller segments or *chunks*.  This is like breaking a textbook chapter into paragraphs so that the system can “digest” them.  Chunking is essential because language models can only process a limited amount of text at once; breaking the text into manageable pieces ensures that important details are not lost.

3. **Embedding the documents** – each chunk is transformed into a numerical vector that captures its meaning.  Embeddings are like fingerprints for text: they allow the computer to measure which passages are most similar to a given query.

4. **Vector store indexing** – all of these vectors are stored in a database designed to support similarity search.  You can think of it as a special index that lets you quickly find passages related to a topic.

5. **Query and retrieval** – when you have a question or item to generate, your query is also embedded and compared against the database to retrieve the most relevant chunks.

6. **Generation with context** – the retrieved text is combined with a large language model (LLM) to produce the assessment item.  Conditioning the model on actual curriculum content helps reduce hallucinations and ensures fidelity to the source material.

7. **Evaluation and refinement** – finally, review and refine the generated items.  Research shows that techniques like key‑point extraction and careful prompting can improve coverage, grammar, and readability of items.

In the following sections, we explore each stage in detail, with code examples using open‑source models.

###

In [1]:
%pip install -q -U langchain langchain-google-genai langchain-chroma langchain-huggingface langchain-text-splitters langchain-community pypdf
# Colab restart is not needed for these packages.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 M

In [2]:
import os

# Keys live in the Colab Secrets panel (key icon, left sidebar), toggled on for this notebook.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("GOOGLE_API_KEY loaded from Colab Secrets.")
except Exception as e:
    print("Could not read Colab Secrets:", type(e).__name__)
    print("Add GOOGLE_API_KEY in the Secrets panel and toggle it on for this notebook.")

# Model IDs in one place -- a one-line fix if anything is retired.
GEMINI_MODEL = "gemini-3.5-flash"
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"

PERSIST_DIR     = "./chroma_curriculum"
COLLECTION_NAME = "grade8_curriculum"

print("Generation model:", GEMINI_MODEL)
print("Embedding model :", EMBED_MODEL, "(runs locally on CPU)")

GOOGLE_API_KEY loaded from Colab Secrets.
Generation model: gemini-3.5-flash
Embedding model : sentence-transformers/all-MiniLM-L6-v2 (runs locally on CPU)


### Stage 1 Get a corpus

**Default: a corpus built into the notebook.** Eight short Grade 8 passages, four science and four
mathematics, each tagged with a source, a standard, a grade, and a subject.

Bring your own PDFs by flipping `USE_OWN_PDFS = True` — the rest of the notebook does not change,
because everything downstream only sees `Document` objects.

In [3]:
from langchain_core.documents import Document

USE_OWN_PDFS = False   # True to use your own PDFs instead

# --- Default path: a small, self-contained Grade 8 corpus -------------------
PASSAGES = [
    {
        "text": (
            "Photosynthesis is the process green plants use to build sugar from carbon dioxide "
            "and water using light energy. It takes place in chloroplasts, organelles that "
            "contain the green pigment chlorophyll. Chlorophyll absorbs light, mostly in the "
            "blue and red parts of the visible spectrum, and reflects green light, which is why "
            "leaves look green. The overall reaction can be summarized as: six carbon dioxide "
            "molecules plus six water molecules, in the presence of light energy, yield one "
            "glucose molecule and six oxygen molecules. The carbon atoms in the glucose come "
            "from carbon dioxide taken in through small openings in the leaf called stomata. "
            "The oxygen released as a by-product comes from water, not from carbon dioxide."
        ),
        "metadata": {"source": "photosynthesis_overview.md", "standard": "MS-LS1-6",
                     "grade": 8, "subject": "science"},
    },
    {
        "text": (
            "Photosynthesis converts light energy into chemical energy stored in the bonds of "
            "glucose. Plants do not create energy; they transform it from one form into another. "
            "Because plants make their own food from inorganic inputs, they are called producers, "
            "or autotrophs. Nearly all food chains begin with producers, so most of the energy "
            "moving through an ecosystem entered it through photosynthesis. A common student "
            "misconception is that plants take in food from the soil through their roots. Roots "
            "take up water and dissolved minerals, but the mass a plant gains as it grows comes "
            "mostly from carbon dioxide in the air. Careful measurement of a potted plant shows "
            "the soil loses very little mass while the plant gains a great deal."
        ),
        "metadata": {"source": "photosynthesis_energy_flow.md", "standard": "MS-LS1-6",
                     "grade": 8, "subject": "science"},
    },
    {
        "text": (
            "Cellular respiration is the process cells use to release energy stored in glucose. "
            "In the presence of oxygen, glucose is broken down into carbon dioxide and water, and "
            "the released energy is captured in ATP, the molecule cells spend to do work. The "
            "summarized reaction is: one glucose molecule plus six oxygen molecules yield six "
            "carbon dioxide molecules, six water molecules, and usable energy. Most of these "
            "reactions occur in the mitochondria. Cellular respiration happens in plant cells as "
            "well as animal cells; plants both make glucose and break it down. During daylight a "
            "plant usually photosynthesizes faster than it respires, so it releases more oxygen "
            "than it consumes."
        ),
        "metadata": {"source": "cellular_respiration_basics.md", "standard": "MS-LS1-7",
                     "grade": 8, "subject": "science"},
    },
    {
        "text": (
            "Photosynthesis and cellular respiration are often described as opposite processes, "
            "but they are not simply reverses of one another. Photosynthesis stores energy; "
            "cellular respiration releases it. The inputs of one are broadly the outputs of the "
            "other: photosynthesis takes in carbon dioxide and water and gives off oxygen, while "
            "cellular respiration takes in oxygen and glucose and gives off carbon dioxide and "
            "water. They occur in different organelles, chloroplasts and mitochondria, and follow "
            "different chemical pathways with different enzymes. Together they cycle matter, "
            "particularly carbon, between organisms and the atmosphere, while energy flows in one "
            "direction: entering as sunlight and leaving as heat."
        ),
        "metadata": {"source": "photosynthesis_respiration_compare.md", "standard": "MS-LS1-7",
                     "grade": 8, "subject": "science"},
    },
    {
        "text": (
            "A linear equation in one variable is solved by keeping the equation balanced: "
            "whatever operation is applied to one side must be applied to the other. To solve "
            "3x + 5 = 20, subtract 5 from both sides to get 3x = 15, then divide both sides by 3 "
            "to get x = 5. Some linear equations have exactly one solution, some have no "
            "solution, and some have infinitely many solutions. If simplifying both sides "
            "produces a false statement such as 4 = 7, the equation has no solution. If it "
            "produces a statement that is always true, such as 6 = 6, then every value of the "
            "variable is a solution."
        ),
        "metadata": {"source": "linear_equations_one_variable.md", "standard": "8.EE.C.7",
                     "grade": 8, "subject": "mathematics"},
    },
    {
        "text": (
            "Equations with variables on both sides are solved by first using the distributive "
            "property to expand any products, then collecting like terms, then isolating the "
            "variable. For 2(x + 4) = 5x - 7, distributing gives 2x + 8 = 5x - 7. Subtracting 2x "
            "from both sides gives 8 = 3x - 7, adding 7 to both sides gives 15 = 3x, and dividing "
            "by 3 gives x = 5. A frequent error is distributing to only the first term inside the "
            "parentheses, producing 2x + 4 instead of 2x + 8. Substituting the answer back into "
            "the original equation is a quick check: 2(5 + 4) = 18 and 5(5) - 7 = 18."
        ),
        "metadata": {"source": "linear_equations_distribute.md", "standard": "8.EE.C.7",
                     "grade": 8, "subject": "mathematics"},
    },
    {
        "text": (
            "The slope-intercept form of a linear function is y = mx + b, where m is the slope "
            "and b is the y-intercept. The slope is the constant rate of change: the amount y "
            "changes for each increase of one in x. The y-intercept is the value of y when x is "
            "0, so it is the point where the graph crosses the vertical axis. In y = 3x - 2, the "
            "slope is 3 and the y-intercept is -2, meaning the line rises 3 units for every 1 "
            "unit to the right and crosses the y-axis at the point (0, -2). A negative slope "
            "means y decreases as x increases, and a slope of 0 gives a horizontal line."
        ),
        "metadata": {"source": "slope_intercept_form.md", "standard": "8.F.B.4",
                     "grade": 8, "subject": "mathematics"},
    },
    {
        "text": (
            "The slope of a line through two points is found by dividing the change in y by the "
            "change in x: m = (y2 - y1) / (x2 - x1). For the points (1, 4) and (3, 10), the slope "
            "is (10 - 4) / (3 - 1) = 6 / 2 = 3. Because a line is straight, any two points on it "
            "give the same slope, which is why slope is a well-defined property of the whole "
            "line rather than of a particular pair of points. A common error is reversing the "
            "order of subtraction in only one of the two differences, for example (10 - 4) / "
            "(1 - 3), which gives the wrong sign. Keep both differences in the same order."
        ),
        "metadata": {"source": "slope_from_two_points.md", "standard": "8.EE.B.6",
                     "grade": 8, "subject": "mathematics"},
    },
]

if not USE_OWN_PDFS:
    docs = [Document(page_content=p["text"], metadata=dict(p["metadata"])) for p in PASSAGES]
    print(f"Built-in corpus: {len(docs)} passages")

# --- Optional path: your own PDFs ------------------------------------------
else:
    from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

    PDF_DIR = "data"
    os.makedirs(PDF_DIR, exist_ok=True)

    # Option A -- upload from your machine (Colab only).
    try:
        from google.colab import files
        uploaded = files.upload()          # pick one or more PDFs
        for name in uploaded:
            os.replace(name, os.path.join(PDF_DIR, name))
    except Exception as e:
        print("Skipping upload widget:", type(e).__name__, "-- put PDFs in ./data instead.")

    # Option B -- whatever is already in ./data.
    loader = DirectoryLoader(PDF_DIR, glob="**/*.pdf", loader_cls=PyPDFLoader)
    docs = loader.load()

    # PyPDFLoader gives us 'source' and 'page'. Add the tags the rest of the notebook expects.
    for d in docs:
        d.metadata["source"] = os.path.basename(d.metadata.get("source", "unknown.pdf"))
        d.metadata.setdefault("page", 0)
        d.metadata.setdefault("standard", "UNSPECIFIED")
        d.metadata.setdefault("grade", 8)
        d.metadata.setdefault("subject", "unspecified")
    print(f"Loaded {len(docs)} pages from {PDF_DIR}/")

print("\nFirst document metadata:", docs[0].metadata)
print("First 120 chars      :", docs[0].page_content[:120], "...")

Built-in corpus: 8 passages

First document metadata: {'source': 'photosynthesis_overview.md', 'standard': 'MS-LS1-6', 'grade': 8, 'subject': 'science'}
First 120 chars      : Photosynthesis is the process green plants use to build sugar from carbon dioxide and water using light energy. It takes ...


## Stage 2 Chunk, with metadata that survives

Splitting is where metadata usually gets lost. It should not: a chunk with no source is a chunk you
cannot cite, filter, or defend. `RecursiveCharacterTextSplitter` copies the parent document's
metadata onto every child chunk, and we add a `chunk_id` so each piece is individually identifiable.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # characters, not tokens
    chunk_overlap=100,     # ~20% -- keeps a sentence that straddles a boundary usable
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True,
)

splits = splitter.split_documents(docs)

# Give every chunk a stable id.
for i, ch in enumerate(splits):
    ch.metadata["chunk_id"] = f"c{i:03d}"

print(f"{len(docs)} documents -> {len(splits)} chunks")
print(f"Mean chunk length: {sum(len(c.page_content) for c in splits) / len(splits):.0f} chars\n")

# Metadata check: did the tags survive the split?
required = {"source", "standard", "grade", "subject", "chunk_id"}
missing = [c.metadata.get("chunk_id") for c in splits if not required.issubset(c.metadata)]
print("Chunks missing required metadata:", missing if missing else "none")

sample = splits[1]
print("\n--- sample chunk ---")
print(sample.page_content)
print("\n--- its metadata ---")
for field, value in sample.metadata.items():
    print(f"  {field}: {value}")

8 documents -> 16 chunks
Mean chunk length: 336 chars

Chunks missing required metadata: none

--- sample chunk ---
. The overall reaction can be summarized as: six carbon dioxide molecules plus six water molecules, in the presence of light energy, yield one glucose molecule and six oxygen molecules. The carbon atoms in the glucose come from carbon dioxide taken in through small openings in the leaf called stomata. The oxygen released as a by-product comes from water, not from carbon dioxide.

--- its metadata ---
  source: photosynthesis_overview.md
  standard: MS-LS1-6
  grade: 8
  subject: science
  start_index: 340
  chunk_id: c001


**Chunk size trade-offs for curriculum material.** Short chunks (200–300 characters) retrieve
precisely but without enough context for the model to write a full item — you get a
definition with no worked example. Long chunks (1,000+) provide context but dilute the embedding, so
a chunk about four different things matches everything weakly and nothing strongly.

For standards and curriculum text, 400–600 characters with 15–25% overlap is a reasonable starting
point: roughly one idea plus its example. Two adjustments worth making for real material:

- If your source is a standards document, consider splitting on the standard itself rather than by
  character count, so one chunk = one standard.
- If passages contain worked examples, raise the overlap so a solution is never separated from the
  problem that set it up.

You will change these numbers in the second `# YOUR TURN` and watch the retrieval score.

## Stage 3 Embed and index

Embeddings run **locally, on this Colab CPU**, with a small sentence-transformers model. No API
call, no key, no quota, no per-token cost and the corpus text never leaves the runtime. For a few
hundred chunks this takes seconds.

The first cell run downloads the model (~90 MB); after that it is cached.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Rebuild from scratch each run so the index always matches the corpus above.
import shutil
shutil.rmtree(PERSIST_DIR, ignore_errors=True)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)

try:
    count = vectorstore._collection.count()
except Exception:
    count = len(splits)
print(f"Indexed {count} chunks into collection '{COLLECTION_NAME}' at {PERSIST_DIR}")

# A stale index is a classic silent failure: you edit the corpus, forget to reindex,
# and retrieve yesterday's text. Deleting the directory first makes that impossible here.

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 16 chunks into collection 'grade8_curriculum' at ./chroma_curriculum


## Stage 4 Retrieve, and check that retrieval is any good

**If retrieval is wrong, nothing downstream can be right.** A generation prompt handed the wrong
passage will still produce a confident, well-formed, plausible item and you will not notice,
because the output looks exactly like the output you wanted. Every RAG failure that survives to
review started as a retrieval failure nobody measured.

So we measure it. Write a handful of queries where you already know which source *should* come
back, run them, and compute **hit@k**: the percentage of queries for which the expected source
appears anywhere in the top *k* results. It is a crude metric. It is also five minutes of work and
it catches the failures that matter.

Ten to twenty query/source pairs, written by whoever knows the corpus, is a usable regression test.
Re-run it every time you change the chunker, the embedding model, or *k*.

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

hits = retriever.invoke("What does the y-intercept mean in slope-intercept form?")

print(f"Retrieved {len(hits)} chunks:\n")
for i, d in enumerate(hits, 1):
    m = d.metadata
    print(f"[{i}] source={m['source']}  standard={m['standard']}  subject={m['subject']}")
    print(f"    {d.page_content[:130]}...\n")

Retrieved 4 chunks:

[1] source=slope_intercept_form.md  standard=8.F.B.4  subject=mathematics
    The slope-intercept form of a linear function is y = mx + b, where m is the slope and b is the y-intercept. The slope is the const...

[2] source=slope_intercept_form.md  standard=8.F.B.4  subject=mathematics
    . A negative slope means y decreases as x increases, and a slope of 0 gives a horizontal line....

[3] source=slope_from_two_points.md  standard=8.EE.B.6  subject=mathematics
    The slope of a line through two points is found by dividing the change in y by the change in x: m = (y2 - y1) / (x2 - x1). For the...

[4] source=linear_equations_one_variable.md  standard=8.EE.C.7  subject=mathematics
    A linear equation in one variable is solved by keeping the equation balanced: whatever operation is applied to one side must be ap...



In [7]:
# --- Retrieval sanity check -------------------------------------------------
# (query, the source that SHOULD come back). Written by hand, from the corpus.

EVAL_PAIRS = [
    ("Where does the oxygen released during photosynthesis come from?",
     "photosynthesis_overview.md"),
    ("Where does the mass a growing plant gains actually come from?",
     "photosynthesis_energy_flow.md"),
    ("Which organelle carries out cellular respiration?",
     "cellular_respiration_basics.md"),
    ("How are photosynthesis and cellular respiration related to each other?",
     "photosynthesis_respiration_compare.md"),
    ("What does it mean if solving an equation gives a false statement like 4 = 7?",
     "linear_equations_one_variable.md"),
    ("How do you solve an equation with variables on both sides?",
     "linear_equations_distribute.md"),
    ("What do m and b stand for in y = mx + b?",
     "slope_intercept_form.md"),
    ("How do you calculate slope from two points on a line?",
     "slope_from_two_points.md"),
]


def hit_at_k(pairs, retr):
    """Return (hit_rate_pct, misses). A hit = expected source anywhere in top-k."""
    misses = []
    n_hit = 0
    for query, expected in pairs:
        got = retr.invoke(query)
        sources = [d.metadata.get("source") for d in got]
        if expected in sources:
            n_hit += 1
        else:
            misses.append({"query": query, "expected": expected, "got": sources})
    return 100.0 * n_hit / len(pairs), misses


rate, misses = hit_at_k(EVAL_PAIRS, retriever)

k = retriever.search_kwargs.get("k", "?")
print(f"hit@{k} = {rate:.0f}%  ({len(EVAL_PAIRS) - len(misses)}/{len(EVAL_PAIRS)} queries)\n")

if misses:
    print("Misses -- these are the queries to look at first:")
    for m in misses:
        print(f"\n  query   : {m['query']}")
        print(f"  expected: {m['expected']}")
        print(f"  got     : {m['got']}")
else:
    print("No misses on this eval set.")
    print("Note: 100% on eight easy queries is a floor, not a guarantee. Add harder ones.")

hit@4 = 100%  (8/8 queries)

No misses on this eval set.
Note: 100% on eight easy queries is a floor, not a guarantee. Add harder ones.


In [8]:
# --- Metadata filtering: retrieve inside one blueprint cell -----------------
# Same query, restricted to science. This is how you keep a math retriever out of
# your science item pool, or target a single standard.

query = "How does energy get stored and released?"

print("UNFILTERED:")
for d in vectorstore.similarity_search(query, k=3):
    print(f"  {d.metadata['subject']:<12} {d.metadata['source']}")

print("\nFILTERED to subject = science:")
for d in vectorstore.similarity_search(query, k=3, filter={"subject": "science"}):
    print(f"  {d.metadata['subject']:<12} {d.metadata['source']}")

print("\nFILTERED to standard = 8.F.B.4:")
for d in vectorstore.similarity_search("rate of change", k=3, filter={"standard": "8.F.B.4"}):
    print(f"  {d.metadata['standard']:<12} {d.metadata['source']}")

# A filtered retriever, ready to drop into the generation chain:
science_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4, "filter": {"subject": "science"}}
)

UNFILTERED:
  science      cellular_respiration_basics.md
  science      photosynthesis_energy_flow.md
  science      photosynthesis_respiration_compare.md

FILTERED to subject = science:
  science      cellular_respiration_basics.md
  science      photosynthesis_energy_flow.md
  science      photosynthesis_respiration_compare.md

FILTERED to standard = 8.F.B.4:
  8.F.B.4      slope_intercept_form.md
  8.F.B.4      slope_intercept_form.md


## Stage 5 Generate a grounded, typed item

Two properties of this stage, both small and both load-bearing.

**Typed, not free text.** The model returns a validated `Item` object, not a paragraph we then
regex apart. If the model omits the key or invents a fifth option, Pydantic raises instead of
letting a malformed item into the pool.

**Cited.** `SourcedItem` extends the canonical `Item` with two fields: which numbered passage
supports the key, and the exact sentence from that passage that supports it. Asking for the quote
not just the passage number is what makes Stage 6 possible. When a model cannot find a supporting
sentence it is likely going to make things up.

In [9]:
from pydantic import BaseModel, Field
from typing import Literal, List


class Option(BaseModel):
    label: Literal["A", "B", "C", "D"]
    text: str = Field(description="Option text, no leading label")
    is_key: bool
    rationale: str = Field(description="Why this option is correct, or the misconception it captures")


class Item(BaseModel):
    stem: str
    options: List[Option]
    key: Literal["A", "B", "C", "D"]
    standard: str = Field(description="Standard or objective the item measures")
    cognitive_level: Literal["recall", "apply", "analyze"]
    target_difficulty: Literal["easy", "medium", "hard"]
    grade: int


class SourcedItem(Item):
    """An Item that carries its evidence with it."""
    source_ids: List[str] = Field(
        description="Numbers of the context passages that support the key, e.g. ['2'] or ['1','3']"
    )
    supporting_quote: str = Field(
        description="A sentence copied verbatim from one of the cited passages that supports the key"
    )


print("Schema ready. SourcedItem fields:", list(SourcedItem.model_fields))

Schema ready. SourcedItem fields: ['stem', 'options', 'key', 'standard', 'cognitive_level', 'target_difficulty', 'grade', 'source_ids', 'supporting_quote']


In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel

llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0.4)


def format_context(retrieved):
    """Number the passages and show their metadata, so the model can cite them."""
    blocks = []
    for i, d in enumerate(retrieved, 1):
        m = d.metadata
        blocks.append(
            f"[{i}] source={m.get('source')} | standard={m.get('standard')} | "
            f"grade={m.get('grade')} | subject={m.get('subject')}\n{d.page_content}"
        )
    return "\n\n".join(blocks)


prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assessment specialist writing Grade 8 multiple-choice items.\n"
     "Write the item ONLY from the numbered passages provided. Do not use outside knowledge, "
     "and do not include content the passages do not contain.\n"
     "Rules:\n"
     "- Exactly four options, labeled A-D, exactly one key.\n"
     "- Every distractor must reflect a plausible student misconception, named in its rationale.\n"
     "- Set 'standard' to the standard shown on the passage you used.\n"
     "- In 'source_ids', list the numbers of the passages that support the key.\n"
     "- In 'supporting_quote', copy one sentence VERBATIM from a cited passage that makes the "
     "key correct. Copy it exactly; do not paraphrase or shorten it."),
    ("human", "Topic: {topic}\n\nPassages:\n{context}"),
])

generate = (
    {"context": retriever | format_context, "topic": lambda x: x}
    | prompt
    | llm.with_structured_output(SourcedItem)
)

# Run retrieval and generation together so we keep the retrieved docs for Stage 6.
grounded_chain = RunnableParallel(docs=retriever, item=generate)

TOPIC = "the meaning of slope in slope-intercept form"

item, retrieved_docs = None, []
try:
    result = grounded_chain.invoke(TOPIC)
    item, retrieved_docs = result["item"], result["docs"]
except Exception as e:
    print("Generation failed:", type(e).__name__)
    print(str(e)[:400])
    print("\nCommon causes: GOOGLE_API_KEY missing or not toggled on for this notebook; "
          "free-tier rate limit (wait ~60s and re-run); model ID retired (see GEMINI_MODEL).")

if item:
    print("STEM:", item.stem, "\n")
    for o in item.options:
        print(f"  {o.label}{'*' if o.is_key else ' '} {o.text}")
        print(f"      -> {o.rationale}")
    print(f"\nKey: {item.key} | standard: {item.standard} | "
          f"level: {item.cognitive_level} | target difficulty: {item.target_difficulty}")
    print(f"\nCITED PASSAGES: {item.source_ids}")
    print(f"SUPPORTING QUOTE: \"{item.supporting_quote}\"")

STEM: In the slope-intercept form of a linear function, what does the slope represent? 

  A  The value of y when x is 0
      -> This option reflects the misconception of confusing the slope with the y-intercept, which is defined as the value of y when x is 0.
  B* The amount y changes for each increase of one in x
      -> According to the passage, the slope is the constant rate of change, which represents the amount y changes for each increase of one in x.
  C  The point where the graph crosses the vertical axis
      -> This option reflects the misconception of confusing the slope with the graphical definition of the y-intercept.
  D  The value of x when the equation is equal to 0
      -> This option reflects the misconception of confusing the slope with the x-intercept or the solution to the equation when y is 0.

Key: B | standard: 8.F.B.4 | level: recall | target difficulty: easy

CITED PASSAGES: ['1']
SUPPORTING QUOTE: "The slope is the constant rate of change: the amount y ch

## Stage 6 Verify grounding

The most common RAG failure is when item key is not actually
supported by the retrieved source, usually because the model read the passages, drifted into its own prior
knowledge, and wrote something true-sounding that the passage never says. Sometimes it is true
anyway. Sometimes it is not. Either way you have lost the traceability you adopted RAG to get.

Two mechanical checks catch most of it:

1. **Does the cited passage exist?** A citation of `[7]` when four passages were retrieved is a
   fabricated citation.
2. **Does the supporting quote actually appear in that passage?** Normalize whitespace, case, and
   quote characters, then do a substring match.

Neither check asks whether the item is *good*. They ask whether it is *grounded*. That is a lower
bar, and it should be automatic, because it costs nothing and it runs on every item.

In [11]:
import re


def normalize(text: str) -> str:
    """Lowercase, collapse whitespace, standardize quotes and dashes."""
    t = text.lower()
    t = t.replace("\u2019", "'").replace("\u2018", "'")
    t = t.replace("\u201c", '"').replace("\u201d", '"')
    t = t.replace("\u2014", "-").replace("\u2013", "-")
    t = re.sub(r"\s+", " ", t)
    return t.strip()


def check_grounding(item, retrieved_docs):
    """Cheap automatic grounding check. Returns pass/fail flags plus what was checked."""
    n = len(retrieved_docs)

    # 1. Do the cited passage numbers exist?
    valid_ids, bad_ids = [], []
    for sid in item.source_ids:
        digits = re.sub(r"[^0-9]", "", str(sid))
        idx = int(digits) if digits else -1
        (valid_ids if 1 <= idx <= n else bad_ids).append(idx if digits else sid)

    citation_valid = bool(valid_ids) and not bad_ids

    # 2. Does the supporting quote appear in a cited passage?
    quote_norm = normalize(item.supporting_quote)
    cited_text = " ".join(normalize(retrieved_docs[i - 1].page_content) for i in valid_ids)
    quote_found = bool(quote_norm) and quote_norm in cited_text

    # Fallback: is the quote anywhere in the retrieved set (right quote, wrong citation)?
    all_text = " ".join(normalize(d.page_content) for d in retrieved_docs)
    quote_in_any = bool(quote_norm) and quote_norm in all_text

    cited_sources = sorted({retrieved_docs[i - 1].metadata.get("source") for i in valid_ids})

    return {
        "citation_valid": citation_valid,
        "quote_found_in_cited": quote_found,
        "quote_found_anywhere": quote_in_any,
        "grounded": citation_valid and quote_found,
        "invalid_ids": bad_ids,
        "cited_sources": cited_sources,
        "n_retrieved": n,
    }


if item:
    report = check_grounding(item, retrieved_docs)
    print("GROUNDING REPORT")
    for k_, v in report.items():
        mark = "" if not isinstance(v, bool) else ("PASS  " if v else "FAIL  ")
        print(f"  {mark}{k_}: {v}")

    if report["grounded"]:
        print("\nItem is grounded: cited passage exists and contains the supporting sentence.")
        print("Provenance:", ", ".join(report["cited_sources"]))
    elif report["quote_found_anywhere"] and not report["quote_found_in_cited"]:
        print("\nThe quote is in the retrieved set but not in the passage cited -- "
              "citation is misnumbered. Route to human review.")
    else:
        print("\nThe supporting sentence is not verbatim in the retrieved passages. "
              "Most often the model paraphrased; occasionally it invented. Either way, "
              "this item does not ship without a reviewer reading the source.")
else:
    print("No item to check -- re-run the generation cell above.")

GROUNDING REPORT
  PASS  citation_valid: True
  PASS  quote_found_in_cited: True
  PASS  quote_found_anywhere: True
  PASS  grounded: True
  invalid_ids: []
  cited_sources: ['slope_intercept_form.md']
  n_retrieved: 4

Item is grounded: cited passage exists and contains the supporting sentence.
Provenance: slope_intercept_form.md


## **Activity**
Insert your own content in the corpus

- Write **two** short passages (80-150 words each) from your content area, add them
to the corpus, rebuild the index, and generate an item from one of them.

**HINT**: fill in MY_PASSAGES, then run this cell straight through. Keep the four metadata keys, the retriever, the filters, and the citation formatter all read them.

In [ ]:

MY_PASSAGES = [
    {
        "text": "REPLACE ME with 80-150 words of your own curriculum or standards text.",
        "metadata": {"source": "my_passage_1.md", "standard": "YOUR.STANDARD",
                     "grade": 8, "subject": "science"},
    },
    {
        "text": "REPLACE ME with a second passage.",
        "metadata": {"source": "my_passage_2.md", "standard": "YOUR.STANDARD",
                     "grade": 8, "subject": "science"},
    },
]

my_docs = [Document(page_content=p["text"], metadata=dict(p["metadata"])) for p in MY_PASSAGES]
my_splits = splitter.split_documents(my_docs)
for j, ch in enumerate(my_splits):
    ch.metadata["chunk_id"] = f"mine{j:03d}"

vectorstore.add_documents(my_splits)
try:
    total = vectorstore._collection.count()
except Exception:
    total = "unknown"
print(f"Added {len(my_splits)} chunks. Collection now holds {total}.")

# Generate from your material. Change the topic to match what you wrote.
MY_TOPIC = "REPLACE ME with a topic your passages cover"

try:
    out = grounded_chain.invoke(MY_TOPIC)
    my_item, my_docs_ret = out["item"], out["docs"]
    print("\nSTEM:", my_item.stem)
    print("Cited:", my_item.source_ids, "| Sources:",
          [d.metadata["source"] for d in my_docs_ret])
    print("Grounding:", check_grounding(my_item, my_docs_ret))
except Exception as e:
    print("Generation failed:", type(e).__name__, str(e)[:300])

## **Activity**
 Move k and chunk size, watch hit@k move
- Change ONE thing at a time and re-run the sanity check. Write down what happens.

**HINT**: k is cheap to change (no reindex). Chunk size is not -- it needs a rebuild, which is why it is a separate function below.

In [ ]:

# --- (a) sweep k -----------------------------------------------------------
for k_try in [1, 2, 4, 8]:
    r = vectorstore.as_retriever(search_kwargs={"k": k_try})
    rate_k, miss_k = hit_at_k(EVAL_PAIRS, r)
    print(f"k={k_try:<2} hit@k = {rate_k:5.1f}%   misses: {len(miss_k)}")

print("\nBigger k almost always raises hit@k. Ask yourself what it costs:")
print("more tokens per call, and more chances for the model to cite a passage")
print("that is topically close but wrong. Hit@k is not free accuracy.\n")


# --- (b) rebuild at a different chunk size ---------------------------------
def rebuild_and_score(chunk_size, chunk_overlap, k=4):
    sp = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = sp.split_documents(docs)
    for i2, c2 in enumerate(chunks):
        c2.metadata["chunk_id"] = f"t{i2:03d}"
    vs = Chroma.from_documents(
        documents=chunks, embedding=embeddings,
        collection_name=f"tmp_{chunk_size}_{chunk_overlap}",
    )   # in-memory, no persist_directory
    rate2, miss2 = hit_at_k(EVAL_PAIRS, vs.as_retriever(search_kwargs={"k": k}))
    print(f"chunk_size={chunk_size:<5} overlap={chunk_overlap:<4} "
          f"chunks={len(chunks):<3} hit@{k} = {rate2:5.1f}%")
    return rate2


for cs, ov in [(200, 40), (500, 100), (1200, 200)]:
    rebuild_and_score(cs, ov)

## When RAG is the wrong tool

**RAG grounds content; it does not guarantee cognitive demand.** You can retrieve exactly the right
passage for MS-LS1-6 and still get a recall item where the standard calls for constructing an
explanation. The passage constrains *what* the item is about, not *what the student has to do with
it*. Alignment to cognitive demand is still a review judgment.

**RAG does not fix bad source material.** If the passage is outdated, wrong, or written above
grade level, RAG will faithfully generate items with those properties and now they carry a
citation, which makes them look more trustworthy than they are.

**RAG adds latency and moving parts.** An embedding model, a vector store, a retriever, and an
index that goes stale the moment the corpus changes. If your source material is small enough to
paste into the prompt one standard, a few paragraphs do that instead. Long-context prompting
beats a retrieval pipeline you have to maintain. Reach for RAG when the corpus is too big for the
context window, changes often, or needs filtering by blueprint.

## Measurement check

**What this technique licenses you to claim.** That the item's *content* was drawn from a source
you selected, and that a named passage supports the key.

**What it does not license.** Nothing about validity: grounding is not evidence that the item
measures the intended construct, and a passed grounding check is not an alignment review.
